This document aims to accomplish the Task 2 "Balancedness: Matched vs Unmatched" questions. It uses the Step 1 store definitions as they are: a store is a WSLCB
**storefront** (address), the universe is the 542 storefronts with a license operating in some month of 2019-2021 by the dated WSLCB samples ([1e'], [6a]), a
storefront-month is **matched** when it has an approved Veraset label that month ([1e]), and a storefront is matched when any of its open months is ([6a]).

(a) Check our store history ([1e'], from the dated WSLCB samples) against the task's store history file (`store_history_2018_2021.zip` in
`raw-data/store-history/`): `store_month_activity_2018_2021.csv` (one row per license and month with positive reported sales, 2018-01 - 2021-12) and
`stores_2018_2021.csv` (535 licenses). Before 2018 the LCB's own archived "Sales Activity by License Number" files (Jul 2014 - Dec 2017, `raw-data/lcb-sales/`)
are used for the opening cohort, and where the two overlap (2018-01 - 2020-08) the archive is a cross-check on the store history file

(i) One row per storefront-month in 2019-2021: open by the WSLCB samples or not, and whether a license operating there had positive sales. Label as ([9ai])

(ii) Licenses with sales in a month when no storefront has them operating (by [1e']), and where those licenses are. Label as ([9aii])

(iii) The store history file's 535 licenses against our 542 storefronts: which are in both, which are only in one, and whether the addresses agree. Label as
([9aiii])

(b) Compare places. Among census places with at least one storefront, compare place characteristics across match-rate quartiles (terciles and quintiles as
robustness checks): population, population density, and urbanicity, median household income, unemployment rate, and poverty rate

(i) One row per place with its match rate and characteristics. Label as ([9bi])

(ii) Characteristics by match-rate group, for terciles, quartiles, and quintiles, with tests. Label as ([9bii])

(c) Compare stores. Matched vs. unmatched storefronts and storefront-months: opening cohort, months active and gap months, sales, address ambiguity (street,
neighborhood, and surrounding retail), and storefront size

(i) One row per storefront with every characteristic. Label as ([9ci])

(ii) Matched vs. unmatched storefronts, side by side, with tests, and a regression with all of them together. Label as ([9cii])

(iii) One row per storefront-month (open months, 2019-2021) and matched vs. unmatched storefront-months, including within-store comparisons. Label as ([9ciii])

(iv) A short list of storefronts to check by hand on Google Maps for storefront size. Label as ([9civ])

Notes:
- Census inputs (downloaded from www2.census.gov, no API key): ACS 2017-2021 5-year tables B19013 (median household income), B23025 (unemployment), B17001
  (poverty) by place; 2020 Census urban blocks (`2020_UA_BLOCKS.txt`); the 2020 block-to-place file (`BlockAssign_ST53_WA.zip`); TIGER/Line 2020 places and
  block groups for land area. Population is the 2020 Census count already in [3d] / `p12_blocks_2020.parquet`
- The ACS 2017-2021 window lines up with the 2019-2021 store window and uses the same 2020 place codes as [3d]
- Sales are self-reported net sales, the LCB's own caveat. In the store history file, 2018-2019 come from the WSLCB's monthly sales-activity-by-license
  reports and 2020-2021 from 502data.com (Internet Archive captures), and only months with positive sales are listed, so "no sales record" means no positive
  sales reported (or not captured)
- Surrounding retail uses the SafeGraph POI file on disk (a current snapshot, not 2019-2021), so it describes today's surroundings
- Charts also get saved to "Task 2/Charts and Figures" as .html files

In [1]:
### IMPORTS AND PRELIMINARIES
import os, re, zipfile, warnings
import numpy as np
import pandas as pd
import polars as pl
import geopandas as gpd
import plotly.express as px
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")

STOREFRONT = ["street_address_norm", "city_norm"]
CENSUS = "raw-data/census"
DOWNLOADS = f"{CENSUS}/downloads"
SALES_FOLDER = "raw-data/lcb-sales"
STORE_HISTORY = "raw-data/store-history/store_history_2018_2021.zip"
POI_FOLDER = "raw-data/veraset-visits-wa/SafeGraph_POI"   # SafeGraph places file on disk (current snapshot)
CHART_FOLDER = "Task 2/Charts and Figures"
os.makedirs(CHART_FOLDER, exist_ok = True)

MONTHS = [f"{y}-{m:02d}" for y in (2019, 2020, 2021) for m in range(1, 13)]
SALES_MONTHS = MONTHS      # the store history file has sales for all of 2019-2021

def show(fig, name):
    fig.show()
    fig.write_html(f"{CHART_FOLDER}/{name}.html", include_plotlyjs = "cdn")

def month_index(m):  # "2019-03" -> months since 2000-01
    return (int(m[:4]) - 2000) * 12 + int(m[5:7]) - 1

In [2]:
### LOADING: Step 1 store tables, used as they are
coverage = pl.read_parquet("processed-data/6a_storefront_coverage.parquet").to_pandas()                 # [6a] 542 storefronts
open_months = pl.read_parquet("processed-data/1e_storefront_months.parquet").filter(~pl.col("current_list_only")).to_pandas()   # [1e']
labels = pl.read_parquet("processed-data/1e_storefront_place_months.parquet")
covered = labels.filter(pl.col("approved")).select(STOREFRONT + ["month"]).unique().to_pandas().assign(covered = True)
locations = pl.read_parquet("processed-data/1a_wslcb_locations.parquet").to_pandas()                    # [1a] license x address
coords = pl.read_parquet("processed-data/1d_wslcb_location_coordinates.parquet").to_pandas()            # [1d]
places = pl.read_parquet("processed-data/3d_place_population_2020.parquet").to_pandas()                 # [3d]

# Storefront-months exactly as in [6a]: open months from [1e'], matched when an approved [1e] label exists that month
store_months = open_months.merge(covered, on = STOREFRONT + ["month"], how = "left")
store_months["covered"] = store_months.covered.fillna(False).astype(bool)
check = store_months.groupby(STOREFRONT).covered.sum().rename("c").reset_index().merge(coverage[STOREFRONT + ["covered_months"]], on = STOREFRONT)
print("storefront-months:", len(store_months), "| covered months agree with [6a]:", (check.c == check.covered_months).all())
print("storefronts:", len(coverage), "| matched:", coverage.matched.sum())

storefront-months: 17742 | covered months agree with [6a]: True
storefronts: 542 | matched: 427


In [3]:
### LOADING: LCB archived monthly retail sales by license (Jul 2014 - Aug 2020), used before 2018 and as a cross-check
## Output: sales_archive, one row per license and month with "sales" (net sales, dollars) and "excise"
def read_sales_recent(path):
    raw = pd.read_excel(path, sheet_name = "Retailers", header = None)
    header = raw.index[raw[0].astype(str).str.strip() == "License Number"][0]
    d = raw.iloc[header + 1:, :4].set_axis(["license", "period", "sales", "excise"], axis = 1)
    d = d[pd.to_numeric(d.license, errors = "coerce").notna()]
    d["month"] = pd.to_datetime(d.period.astype(str).str[:7], format = "%Y-%m").dt.strftime("%Y-%m")
    return d

def read_sales_old(path):  # one sheet, producers / processors / retailers blocks; retail rows follow the "RETAILERS" heading
    raw = pd.read_excel(path, header = None)
    start = raw.index[raw[0].astype(str).str.strip() == "RETAILERS"][0]
    d = raw.iloc[start:, :4].set_axis(["license", "period", "sales", "excise"], axis = 1)
    d = d[pd.to_numeric(d.license, errors = "coerce").notna()]
    d["month"] = pd.to_datetime(d.period.astype(str).str.split(" - ").str[0], format = "%m/%d/%Y").dt.strftime("%Y-%m")
    return d

sa = pd.concat([read_sales_old(f"{SALES_FOLDER}/tax_by_license_thru_2017-10-31.xlsx"),
                read_sales_recent(f"{SALES_FOLDER}/sales_by_license_2020-10-02.xlsx")])
sa["license"] = sa.license.astype(int)
sa["sales"] = pd.to_numeric(sa.sales, errors = "coerce")
sa["excise"] = pd.to_numeric(sa.excise, errors = "coerce")
sales_archive = sa.groupby(["license", "month"], as_index = False)[["sales", "excise"]].sum()   # a few license-months are reported in two rows
print("archive license-months:", len(sales_archive), "| licenses:", sales_archive.license.nunique(), "| months:", sales_archive.month.min(), "-", sales_archive.month.max())

archive license-months: 23669 | licenses: 563 | months: 2014-07 - 2020-08


In [4]:
### LOADING: the task's store history file (2018-01 - 2021-12)
## Output: history_months, one row per license and month with positive reported sales; history_stores, one row per license (535)
##         sales: what the rest of the notebook uses, the store history file from 2018-01 on and the LCB archive before
with zipfile.ZipFile(STORE_HISTORY) as z:
    history_months = pd.read_csv(z.open("store_history_2018_2021/store_month_activity_2018_2021.csv"), dtype = {"license": str})
    history_stores = pd.read_csv(z.open("store_history_2018_2021/stores_2018_2021.csv"), dtype = {"license": str, "zip5": str, "ubi": str})
history_months["license"] = history_months.license.astype(int)
history_stores["license"] = history_stores.license.astype(int)
history_months = history_months.rename(columns = {"period": "month", "total_sales": "sales", "excise_tax": "excise"})
print("store history license-months:", len(history_months), "| licenses:", history_months.license.nunique(), "| months:", history_months.month.min(), "-",
      history_months.month.max(), "| duplicates:", history_months.duplicated(["license", "month"]).sum(), "| zero or negative:", (history_months.sales <= 0).sum())
print(history_months.groupby(history_months.month.str[:4]).data_source.first())

sales = pd.concat([sales_archive[sales_archive.month < "2018-01"], history_months[["license", "month", "sales", "excise"]]], ignore_index = True)

## Cross-check where both exist (2018-01 - 2020-08): does the store history file have the same license-months with positive sales as the LCB archive?
ov = (sales_archive[(sales_archive.month >= "2018-01") & (sales_archive.sales > 0)][["license", "month", "sales"]]
      .merge(history_months[history_months.month <= "2020-08"][["license", "month", "sales"]], on = ["license", "month"], how = "outer",
             suffixes = ("_archive", "_history"), indicator = True))
ov["year"] = ov.month.str[:4]
print("\nLicense-months with positive sales, 2018-01 - 2020-08: in both / archive only / store history only")
display(pd.crosstab(ov.year, ov._merge))
b = ov[ov._merge == "both"]
print(f"in both: sales within 1%: {(abs(b.sales_history / b.sales_archive - 1) < 0.01).mean():.1%} | correlation {b.sales_history.corr(b.sales_archive):.4f}")

store history license-months: 20874 | licenses: 535 | months: 2018-01 - 2021-12 | duplicates: 0 | zero or negative: 0
month
2018    WSLCB monthly sales-activity-by-license (self-...
2019    WSLCB monthly sales-activity-by-license (self-...
2020    502data.com monthly sales by licence (Internet...
2021    502data.com monthly sales by licence (Internet...
Name: data_source, dtype: str

License-months with positive sales, 2018-01 - 2020-08: in both / archive only / store history only


_merge,left_only,right_only,both
year,,,
2018,3,5,4999
2019,3,5,5197
2020,0,64,3444


in both: sales within 1%: 99.9% | correlation 0.9998


### PART A: our store history ([1e']) against the store history file

In [5]:
### PART A, SUBPART I
## Output: [9ai] one row per storefront and month in 2019-2021 that is open by [1e'] or has sales by a license that operates at the storefront in some month
##         Columns are STOREFRONT, "month", "open_wslcb" (a license is operating there that month by the dated WSLCB samples), "sales" (net sales of the
##         licenses operating there that month, or of the storefront's licenses for months it isn't open), "has_sales" (sales > 0), "covered" (approved
##         Veraset label that month), "matched" (the storefront is matched in [6a]), "status": "open, sales", "open, no sales record", "sales, not open by WSLCB"
## Name: 9ai_store_history_check.parquet
lic_store = open_months[STOREFRONT + ["month", "operating_licenses"]].explode("operating_licenses").rename(columns = {"operating_licenses": "license"})
lic_store["license"] = lic_store.license.astype(int)
both = lic_store.groupby(["license", "month"]).size()
print("license-months operating at 2+ storefronts at once (sales split equally):", (both > 1).sum())
lic_store = lic_store.merge(both.rename("n_storefronts").reset_index(), on = ["license", "month"])

s = sales[sales.month.isin(SALES_MONTHS)]
open_sales = (lic_store[lic_store.month.isin(SALES_MONTHS)]
    .merge(s, on = ["license", "month"], how = "left")
    .assign(sales = lambda d: d.sales / d.n_storefronts)
    .groupby(STOREFRONT + ["month"], as_index = False)
    .agg(sales = ("sales", lambda x: x.sum(min_count = 1)))
    .assign(open_wslcb = True))

# Months a storefront's own licenses (any license ever operating there) had sales, but no license is operating there by [1e']
store_licenses = lic_store[STOREFRONT + ["license"]].drop_duplicates()
lic_n_stores = store_licenses.groupby("license").size().rename("n_storefronts_ever")
extra = (store_licenses.merge(lic_n_stores, on = "license")
    .query("n_storefronts_ever == 1")     # only licenses that never move, so the sales are at this address
    .merge(s[s.sales > 0], on = "license")
    .merge(open_sales[STOREFRONT + ["month"]], on = STOREFRONT + ["month"], how = "left", indicator = True)
    .query("_merge == 'left_only'")
    .groupby(STOREFRONT + ["month"], as_index = False).agg(sales = ("sales", "sum"))
    .assign(open_wslcb = False))

history_check = (pd.concat([open_sales, extra])
    .merge(store_months[STOREFRONT + ["month", "covered"]], on = STOREFRONT + ["month"], how = "left")
    .merge(coverage[STOREFRONT + ["matched"]], on = STOREFRONT, how = "left"))
history_check["covered"] = history_check.covered.fillna(False).astype(bool)
history_check["has_sales"] = history_check.sales.fillna(0) > 0
history_check["status"] = np.select(
    [history_check.open_wslcb & history_check.has_sales, history_check.open_wslcb],
    ["open, sales", "open, no sales record"], "sales, not open by WSLCB")
history_check.to_parquet("processed-data/9ai_store_history_check.parquet")

t = history_check.groupby("status").agg(storefront_months = ("month", "size"), storefronts = ("street_address_norm", lambda x: x.nunique()))
t["share"] = t.storefront_months / t.storefront_months.sum()
display(t)
print("\nBy matched status of the storefront (share of its storefront-months):")
display(pd.crosstab(history_check.matched, history_check.status, normalize = "index").round(3))
print("\nStorefronts open by WSLCB in 2019-2021 with no sales record in any of those months:")
no_sales_ever = history_check[history_check.open_wslcb].groupby(STOREFRONT).has_sales.any()
print((~no_sales_ever).sum(), "of", len(no_sales_ever))

## What the match rate would be with the universe gated by sales: drop storefronts open in 2019-2021 with no sales in any of those months
gate = coverage.merge(no_sales_ever.rename("any_sales").reset_index(), on = STOREFRONT, how = "left")
gate["kept"] = gate.any_sales.fillna(True).astype(bool)
print(f"storefronts kept: {gate.kept.sum()} of {len(gate)} | match rate {gate.matched.mean():.1%} -> {gate[gate.kept].matched.mean():.1%}"
      f" | dropped: {(~gate.kept & gate.matched).sum()} matched, {(~gate.kept & ~gate.matched).sum()} unmatched")
hm = history_check[history_check.open_wslcb]
print(f"storefront-months 2019-2021 covered: all open {hm.covered.mean():.1%}, open with sales {hm[hm.has_sales].covered.mean():.1%}")
print("\nMatched storefronts with no sales in any open month of 2019-2021:")
display(gate[gate.matched & ~gate.kept][STOREFRONT + ["tradenames", "licenses", "open_months", "covered_months"]])

license-months operating at 2+ storefronts at once (sales split equally): 87


,storefront_months,storefronts,share
status,,,
"open, no sales record",2056,199,0.115512
"open, sales",15686,492,0.881285
"sales, not open by WSLCB",57,7,0.003202



By matched status of the storefront (share of its storefront-months):


status,"open, no sales record","open, sales","sales, not open by WSLCB"
matched,,,
False,0.519,0.477,0.005
True,0.036,0.961,0.003



Storefronts open by WSLCB in 2019-2021 with no sales record in any of those months:
50 of 542
storefronts kept: 492 of 542 | match rate 78.8% -> 86.6% | dropped: 1 matched, 49 unmatched
storefront-months 2019-2021 covered: all open 61.7%, open with sales 68.4%

Matched storefronts with no sales in any open month of 2019-2021:


,street_address_norm,city_norm,tradenames,licenses,open_months,covered_months
160,17903 HWY 536,MOUNT VERNON,"[CANNAZONE, FWEEDOM CANNABIS]",[422000],24,18


In [6]:
### PART A, SUBPART II
## Output: [9aii] licenses with positive retail sales in some month of 2019-2021 when no storefront has them operating by [1e']
##         Columns are "license", "months_with_sales_not_operating", "first_month", "last_month", "sales" (total in those months), "in_1a" (the license is in
##         [1a] at all), "storefronts_in_1a" (its addresses in [1a]), "any_storefront_in_universe" (one of those addresses is one of the 542 storefronts)
## Name: 9aii_sales_without_operating_license.parquet
operating = lic_store[["license", "month"]].drop_duplicates().assign(op = True)
orphan = (s[s.sales > 0].merge(operating, on = ["license", "month"], how = "left"))
orphan = orphan[orphan.op.isna()]
addresses = locations.assign(storefront = locations.street_address_norm + ", " + locations.city_norm).groupby("license").storefront.apply(lambda x: sorted(set(x)))
universe = set(coverage.street_address_norm + ", " + coverage.city_norm)
orphans = (orphan.groupby("license").agg(months_with_sales_not_operating = ("month", "size"), first_month = ("month", "min"),
                                         last_month = ("month", "max"), sales = ("sales", "sum")).reset_index())
orphans["storefronts_in_1a"] = orphans.license.map(addresses)
orphans["in_1a"] = orphans.storefronts_in_1a.notna()
orphans["any_storefront_in_universe"] = orphans.storefronts_in_1a.apply(lambda x: isinstance(x, list) and any(a in universe for a in x))
orphans.to_parquet("processed-data/9aii_sales_without_operating_license.parquet")

total = s[s.sales > 0]
print(f"license-months with sales: {len(total)}; with no storefront operating them by [1e']: {len(orphan)} ({len(orphan) / len(total):.1%}),"
      f" {orphan.sales.sum() / total.sales.sum():.1%} of sales")
print("those licenses:", len(orphans), "| not in [1a] at all:", (~orphans.in_1a).sum(),
      "| in [1a], an address in the 542:", orphans.any_storefront_in_universe.sum(),
      "| in [1a], no address in the 542:", (orphans.in_1a & ~orphans.any_storefront_in_universe).sum())
display(orphans.sort_values("months_with_sales_not_operating", ascending = False).head(15))

license-months with sales: 15870; with no storefront operating them by [1e']: 145 (0.9%), 0.4% of sales
those licenses: 19 | not in [1a] at all: 0 | in [1a], an address in the 542: 11 | in [1a], no address in the 542: 8


,license,months_with_sales_not_operating,first_month,last_month,sales,storefronts_in_1a,in_1a,any_storefront_in_universe
2,413846,14,2019-01,2020-03,91343.07,"[551 14TH AVE, LONGVIEW, 946 WASHINGTON WAY, L...",True,True
5,420477,13,2020-11,2021-11,1091498.00,"[7217 212TH ST SW, EDMONDS]",True,True
7,420800,10,2019-01,2019-10,160046.70,"[141 SAMISH WAY, BELLINGHAM, 2119 LINCOLN ST, ...",True,True
6,420785,9,2021-04,2021-12,3770521.00,"[5917 N 26TH ST, TACOMA]",True,True
9,421632,9,2021-04,2021-12,1424712.00,"[1123 SLEATER KINNEY RD SE, LACEY]",True,True
4,415183,9,2021-04,2021-12,1476450.00,"[1428 OLYMPIC HWY, SHELTON, 3005 NORTHVIEW CIR...",True,True
3,414454,9,2021-04,2021-12,452617.00,"[19001 VASHON HWY SW, VASHON, 19001 VASHON HWY...",True,True
14,431909,9,2021-04,2021-12,592295.00,"[311 OAK ST, KELSO]",True,False
12,429212,9,2021-04,2021-12,10356.00,"[1410 NW GILMAN BLVD, ISSAQUAH, 230 NE JUNIPER...",True,True
13,430876,8,2021-05,2021-12,1935322.00,"[1207 1209 E WASHINGTON AVE, UNION GAP, 1209 E...",True,False


In [7]:
### PART A, SUBPART III
## Output: [9aiii] one row per license in the store history file (535) or operating at one of the 542 storefronts in 2019-2021 by [1e']
##         Columns are "license", "in_store_history", "in_1e_prime" (operating at a universe storefront in some month of 2019-2021), "history_name",
##         "history_address", "history_city", STOREFRONT (the license's [1e'] storefront in its last operating month), "same_street_number" (the store history
##         address and the [1e'] storefront start with the same number), "same_city", "history_first_active", "history_last_active", "history_months_active"
## Name: 9aiii_store_history_licenses.parquet
last_store = lic_store.sort_values("month").groupby("license").last()[STOREFRONT].reset_index()
hs = history_stores[["license", "name", "address", "city", "first_active_month", "last_active_month", "n_months_active"]].rename(
    columns = {"name": "history_name", "address": "history_address", "city": "history_city", "first_active_month": "history_first_active",
               "last_active_month": "history_last_active", "n_months_active": "history_months_active"})
lic_rec = hs.assign(in_store_history = True).merge(last_store.assign(in_1e_prime = True), on = "license", how = "outer")
lic_rec[["in_store_history", "in_1e_prime"]] = lic_rec[["in_store_history", "in_1e_prime"]].fillna(False).astype(bool)
num = lambda x: x.astype(str).str.extract(r"^\s*(\d+)")[0]
both_l = lic_rec.in_store_history & lic_rec.in_1e_prime
lic_rec["same_street_number"] = np.where(both_l, num(lic_rec.history_address) == num(lic_rec.street_address_norm), np.nan)
lic_rec["same_city"] = np.where(both_l, lic_rec.history_city.astype(str).str.upper().str.strip() == lic_rec.city_norm.astype(str).str.upper().str.strip(), np.nan)
lic_rec.to_parquet("processed-data/9aiii_store_history_licenses.parquet")

print("licenses: in both", both_l.sum(), "| store history only", (lic_rec.in_store_history & ~lic_rec.in_1e_prime).sum(),
      "| [1e'] only", (~lic_rec.in_store_history & lic_rec.in_1e_prime).sum())
print(f"in both: same street number {lic_rec[both_l].same_street_number.astype(bool).mean():.1%}, same city {lic_rec[both_l].same_city.astype(bool).mean():.1%}")
print("\nStore history licenses not operating at any of the 542 storefronts in 2019-2021 by [1e']:")
display(lic_rec[lic_rec.in_store_history & ~lic_rec.in_1e_prime][["license", "history_name", "history_address", "history_city", "history_first_active",
                                                                    "history_last_active", "history_months_active"]].sort_values("history_months_active", ascending = False))
print("\nLicenses operating at one of the 542 by [1e'] but not in the store history file, by whether the storefront is matched:")
nh = lic_rec[~lic_rec.in_store_history & lic_rec.in_1e_prime].merge(coverage[STOREFRONT + ["matched"]], on = STOREFRONT, how = "left")
display(nh.matched.value_counts())
st_in_hist = lic_store.merge(lic_rec[lic_rec.in_store_history][["license"]], on = "license").groupby(STOREFRONT).size().rename("n").reset_index()
cov_h = coverage.merge(st_in_hist, on = STOREFRONT, how = "left")
print(f"storefronts with no license in the store history file: {cov_h.n.isna().sum()} of {len(cov_h)}"
      f" ({cov_h[cov_h.n.isna()].matched.sum()} matched, {(cov_h.n.isna() & ~cov_h.matched).sum()} unmatched)")
print("\nAddress disagreements (licenses in both, different street number):")
display(lic_rec[both_l & (lic_rec.same_street_number == False)][["license", "history_name", "history_address", "history_city"] + STOREFRONT])

licenses: in both 498 | store history only 37 | [1e'] only 75
in both: same street number 99.4%, same city 99.8%

Store history licenses not operating at any of the 542 storefronts in 2019-2021 by [1e']:


,license,history_name,history_address,history_city,history_first_active,history_last_active,history_months_active
117,413846,KING CRONIC 3,946 WASHINGTON WAY,LONGVIEW,2018-01,2020-03,25.0
312,420800,THE POT SHOP BELLINGHAM,2119 LINCOLN STREET,BELLINGHAM,2018-01,2019-10,22.0
32,368015,THC OF OLYMPIA,5628 OLD HWY 410 SW STE A,OLYMPIA,2018-02,2018-12,11.0
212,414889,HIGHWAY 7,7046 PACIFIC AVE,TACOMA,2018-01,2018-10,10.0
336,421490,NORTHWEST COLLECTIVE,9326 KIMMIE ST SW STE B,OLYMPIA,2018-01,2018-10,10.0
421,422507,NIRVANA CANNABIS COMPANY,11980 W SUNSET HWY STE A,AIRWAY HEIGHTS,2018-01,2018-10,10.0
474,423977,HIGH SOCIETY TACOMA,3906 S 12TH ST,TACOMA,2018-01,2018-10,10.0
469,423775,21+ RECREATIONAL MARIJUANA,3062 SW HWY 16 STE A,BREMERTON,2018-01,2018-09,9.0
118,413847,SWEET GREENS NORTHWEST,19943 OLD HWY 99 SW,ROCHESTER,2018-01,2018-09,9.0
342,421568,BUD HUT,1717 NE ANDRESEN RD,VANCOUVER,2018-01,2018-09,9.0



Licenses operating at one of the 542 by [1e'] but not in the store history file, by whether the storefront is matched:


matched
False    49
True     26
Name: count, dtype: int64

storefronts with no license in the store history file: 46 of 542 (1 matched, 45 unmatched)

Address disagreements (licenses in both, different street number):


,license,history_name,history_address,history_city,street_address_norm,city_norm
59,413314,2020 SOLUTIONS ON THE GUIDE,1706 MT BAKER HWY,BELLINGHAM,5655 GUIDE MERIDIAN,BELLINGHAM
343,421577,PLAUSIBLE PRODUCTS,7829 LEARY WAY NE,REDMOND,8296 AVONDALE WAY NE,REDMOND
351,421667,GREEN LADY EVERETT,14221 LAKE RD,LYNNWOOD,18026 HWY 9 SE,SNOHOMISH


In [8]:
### PART A, CHART
monthly = history_check.groupby(["month", "status"]).size().rename("storefronts").reset_index()
fig = px.bar(monthly, x = "month", y = "storefronts", color = "status", title = "Storefronts by month: WSLCB samples vs. store history sales records (2019-2021)",
             category_orders = {"status": ["open, sales", "open, no sales record", "sales, not open by WSLCB"]})
show(fig, "wslcb_open_vs_store_history_sales_by_month")

### PART B: places

In [9]:
### LOADING: place characteristics
## ACS 2017-2021 5-year, by place (summary level 160). "-" and the ACS jam values (e.g. -666666666) mean no estimate
def acs(table, cols):
    d = pd.read_csv(f"{DOWNLOADS}/acsdt5y2021-{table}.dat", sep = "|", dtype = str, usecols = ["GEO_ID"] + cols)
    d = d[d.GEO_ID.str.startswith("1600000US53")].copy()
    d["place_fips"] = d.GEO_ID.str[-7:]
    for c in cols:
        d[c] = pd.to_numeric(d[c], errors = "coerce")
        d.loc[d[c] < 0, c] = np.nan
    return d.drop(columns = "GEO_ID")

acs_place = (acs("b19013", ["B19013_E001"])
    .merge(acs("b23025", ["B23025_E003", "B23025_E005"]), on = "place_fips", how = "outer")
    .merge(acs("b17001", ["B17001_E001", "B17001_E002"]), on = "place_fips", how = "outer"))
acs_place["median_hh_income"] = acs_place.B19013_E001
acs_place["unemployment_rate"] = acs_place.B23025_E005 / acs_place.B23025_E003.where(acs_place.B23025_E003 > 0)
acs_place["poverty_rate"] = acs_place.B17001_E002 / acs_place.B17001_E001.where(acs_place.B17001_E001 > 0)

## Land area: TIGER/Line 2020 places
place_shapes = gpd.read_file(f"zip://{DOWNLOADS}/tl_2020_53_place.zip")
place_area = pd.DataFrame({"place_fips": place_shapes.GEOID, "land_sq_mi": place_shapes.ALAND / 2589988.11})

## Urbanicity: share of the place's 2020 population living in 2020 Census urban blocks
blocks = pl.read_parquet(f"{CENSUS}/p12_blocks_2020.parquet").to_pandas()
urban_blocks = pd.read_csv(f"{DOWNLOADS}/2020_UA_BLOCKS.txt", sep = "|", dtype = str, usecols = ["STATE", "GEOID"])
urban_blocks = set(urban_blocks[urban_blocks.STATE == "53"].GEOID)
blocks["urban"] = blocks.block2020.isin(urban_blocks)
with zipfile.ZipFile(f"{DOWNLOADS}/BlockAssign_ST53_WA.zip") as z:
    block_place = pd.read_csv(z.open("BlockAssign_ST53_WA_INCPLACE_CDP.txt"), sep = "|", dtype = str)
block_place = block_place.dropna(subset = ["PLACEFP"]).assign(place_fips = lambda d: "53" + d.PLACEFP).rename(columns = {"BLOCKID": "block2020"})
urban_place = (blocks.merge(block_place[["block2020", "place_fips"]], on = "block2020")
    .assign(urban_pop = lambda d: d.population_total * d.urban)
    .groupby("place_fips").agg(pop_blocks = ("population_total", "sum"), urban_pop = ("urban_pop", "sum")).reset_index())
urban_place["urban_share"] = urban_place.urban_pop / urban_place.pop_blocks.where(urban_place.pop_blocks > 0)

place_chars = (places[["place_fips", "place_name", "place_type", "population_total"]]
    .merge(place_area, on = "place_fips", how = "left")
    .merge(urban_place[["place_fips", "pop_blocks", "urban_share"]], on = "place_fips", how = "left")
    .merge(acs_place[["place_fips", "median_hh_income", "unemployment_rate", "poverty_rate"]], on = "place_fips", how = "left"))
place_chars["pop_density"] = place_chars.population_total / place_chars.land_sq_mi
print("places:", len(place_chars), "| block sums equal [3d] population:", (place_chars.pop_blocks == place_chars.population_total).mean().round(3))
print("missing:", place_chars[["land_sq_mi", "urban_share", "median_hh_income", "unemployment_rate", "poverty_rate"]].isna().sum().to_dict())

places: 639 | block sums equal [3d] population: 1.0
missing: {'land_sq_mi': 0, 'urban_share': 0, 'median_hh_income': 58, 'unemployment_rate': 13, 'poverty_rate': 5}


In [10]:
### PART B, SUBPART I
## Output: [9bi] one row per census place with at least one of the 542 storefronts
##         Columns are "place_fips", "place_name", "place_type", "storefronts", "matched", "match_rate" (matched / storefronts, the main measure),
##         "open_months", "covered_months", "month_coverage" (covered / open months, an alternative with fewer ties), and the characteristics:
##         "population_total" (2020 Census), "land_sq_mi", "pop_density" (people per square mile), "urban_share" (share of people in 2020 urban blocks),
##         "median_hh_income", "unemployment_rate", "poverty_rate" (ACS 2017-2021)
## Name: 9bi_place_match_rates.parquet
place_rates = (coverage[coverage.place_fips.notna()]
    .groupby("place_fips").agg(storefronts = ("matched", "size"), matched = ("matched", "sum"),
                               open_months = ("open_months", "sum"), covered_months = ("covered_months", "sum")).reset_index())
place_rates["match_rate"] = place_rates.matched / place_rates.storefronts
place_rates["month_coverage"] = place_rates.covered_months / place_rates.open_months
place_rates = place_rates.merge(place_chars, on = "place_fips", how = "left")
place_rates.to_parquet("processed-data/9bi_place_match_rates.parquet")

print("places with a storefront:", len(place_rates), "| storefronts in them:", place_rates.storefronts.sum(),
      "| storefronts outside any place (not in Part B):", coverage.place_fips.isna().sum())
print("places by number of storefronts:", place_rates.storefronts.value_counts().sort_index().to_dict())
print("match rate: share of places at exactly 0:", (place_rates.match_rate == 0).mean().round(3), "| at exactly 1:", (place_rates.match_rate == 1).mean().round(3),
      "| distinct values:", place_rates.match_rate.nunique())

places with a storefront: 170 | storefronts in them: 466 | storefronts outside any place (not in Part B): 76
places by number of storefronts: {1: 78, 2: 41, 3: 21, 4: 7, 5: 10, 6: 5, 7: 3, 12: 1, 15: 1, 16: 2, 55: 1}
match rate: share of places at exactly 0: 0.165 | at exactly 1: 0.6 | distinct values: 13


In [11]:
### PART B, SUBPART II
## Output: [9bii] place characteristics by match-rate group, for terciles, quartiles, and quintiles of the match rate (and of month coverage), all places and
##         places with 3+ storefronts
##         Groups never split tied match rates: each place's group comes from its average rank, so tied places land together and some groups can end up
##         empty or merged (the "groups" column says how many there really are). Group 1 = lowest match rate
##         Columns are "rate", "places_kept", "k", "groups", "group", "places", "storefronts", "rate_min", "rate_max", then the mean and median of each
##         characteristic, and "kruskal_p_<characteristic>" (Kruskal-Wallis test of equal distributions across the groups)
## Name: 9bii_place_characteristics_by_group.parquet
CHARS = {"population_total": "Population (2020)", "pop_density": "People per sq. mile", "urban_share": "Share urban",
         "median_hh_income": "Median household income", "unemployment_rate": "Unemployment rate", "poverty_rate": "Poverty rate"}

def tie_groups(x, k):
    g = np.ceil(x.rank(method = "average") / len(x) * k).clip(1, k).astype(int)
    return g.map({v: i + 1 for i, v in enumerate(sorted(g.unique()))})

rows = []
for rate in ["match_rate", "month_coverage"]:
    for kept, d0 in [("all", place_rates), ("3+ storefronts", place_rates[place_rates.storefronts >= 3])]:
        for k in [3, 4, 5]:
            d = d0.assign(group = tie_groups(d0[rate], k))
            pvals = {f"kruskal_p_{c}": stats.kruskal(*[g[c].dropna() for _, g in d.groupby("group")]).pvalue if d.group.nunique() > 1 else np.nan for c in CHARS}
            for grp, g in d.groupby("group"):
                r = {"rate": rate, "places_kept": kept, "k": k, "groups": d.group.nunique(), "group": grp, "places": len(g), "storefronts": g.storefronts.sum(),
                     "rate_min": g[rate].min(), "rate_max": g[rate].max()}
                for c in CHARS:
                    r[f"mean_{c}"], r[f"median_{c}"] = g[c].mean(), g[c].median()
                rows.append({**r, **pvals})
by_group = pd.DataFrame(rows)
by_group.to_parquet("processed-data/9bii_place_characteristics_by_group.parquet")

main = by_group.query("rate == 'match_rate' and places_kept == 'all' and k == 4")
print("Main version (storefront match rate, all places, quartiles):")
display(main[["group", "places", "storefronts", "rate_min", "rate_max"] + [f"median_{c}" for c in CHARS]].round(3))
display(main[[f"kruskal_p_{c}" for c in CHARS]].iloc[[0]].round(3))

## Spearman correlation of each characteristic with the match rate across places (no grouping, so ties don't matter), weighted equally
corr = []
for rate in ["match_rate", "month_coverage"]:
    for kept, d0 in [("all", place_rates), ("3+ storefronts", place_rates[place_rates.storefronts >= 3])]:
        for c in CHARS:
            ok = d0[[rate, c]].dropna()
            rho, p = stats.spearmanr(ok[rate], ok[c])
            corr.append({"rate": rate, "places_kept": kept, "characteristic": c, "places": len(ok), "spearman_rho": rho, "p_value": p})
corr = pd.DataFrame(corr)
display(corr.pivot_table(index = "characteristic", columns = ["rate", "places_kept"], values = ["spearman_rho", "p_value"]).round(3))

## Storefront level, no grouping: characteristics of the place each matched / unmatched storefront is in (storefronts outside any place left out)
sp = coverage[coverage.place_fips.notna()][["place_fips", "matched"]].merge(place_chars, on = "place_fips", how = "left")
store_level = pd.DataFrame([{"characteristic": c, "matched_median": sp[sp.matched][c].median(), "unmatched_median": sp[~sp.matched][c].median(),
                             "matched_mean": sp[sp.matched][c].mean(), "unmatched_mean": sp[~sp.matched][c].mean(),
                             "ranksum_p": stats.mannwhitneyu(sp[sp.matched][c].dropna(), sp[~sp.matched][c].dropna()).pvalue} for c in CHARS])
print("\nPlace characteristics of matched vs. unmatched storefronts (storefront level, %d storefronts in places):" % len(sp))
display(store_level.round(3))

Main version (storefront match rate, all places, quartiles):


,group,places,storefronts,rate_min,rate_max,median_population_total,median_pop_density,median_urban_share,median_median_hh_income,median_unemployment_rate,median_poverty_rate
3,1,45,77,0.000,0.500,10148.0,1715.967,0.988,73131.0,0.056,0.107
4,2,23,200,0.667,0.875,73256.0,3736.587,1.000,71357.0,0.051,0.114
5,3,102,189,1.000,1.000,7161.0,1478.068,0.989,69028.0,0.051,0.090


,kruskal_p_population_total,kruskal_p_pop_density,kruskal_p_urban_share,kruskal_p_median_hh_income,kruskal_p_unemployment_rate,kruskal_p_poverty_rate
3,0.0,0.0,0.01,0.731,0.579,0.252


p_value                                spearman_rho  \
rate                  match_rate        month_coverage            match_rate   
places_kept       3+ storefronts    all 3+ storefronts    all 3+ storefronts   
characteristic                                                                 
median_hh_income           0.481  0.426          0.969  0.686         -0.101   
pop_density                0.178  0.114          0.379  0.460         -0.192   
population_total           0.337  0.012          0.574  0.970         -0.137   
poverty_rate               0.987  0.392          0.271  0.394         -0.002   
unemployment_rate          0.493  0.391          0.156  0.451         -0.098   
urban_share                0.783  0.744          0.921  0.134         -0.039   

                                                
rate                     month_coverage         
places_kept          all 3+ storefronts    all  
characteristic                                  
median_hh_income  -0.062         -0.006  0.032  
pop_density       -0.122         -0.126  0.057  
population_total  -0.193         -0.081  0.003  
poverty_rate      -0.066         -0.157 -0.066  
unemployment_rate -0.067         -0.202 -0.059  
urban_share       -0.025          0.014  0.116


Place characteristics of matched vs. unmatched storefronts (storefront level, 466 storefronts in places):


,characteristic,matched_median,unmatched_median,matched_mean,unmatched_mean,ranksum_p
0,population_total,27729.000,23569.000,136818.512,116785.697,0.408
1,pop_density,3250.850,3111.961,3585.612,3386.564,0.574
2,urban_share,0.999,0.999,0.905,0.795,0.469
3,median_hh_income,71357.000,72969.000,77872.728,80685.186,0.491
4,unemployment_rate,0.050,0.052,0.055,0.054,0.671
5,poverty_rate,0.106,0.107,0.118,0.122,0.947


In [12]:
### PART B, CHART: medians by match-rate quartile
long = main.melt(id_vars = ["group", "rate_min", "rate_max", "places"], value_vars = [f"median_{c}" for c in CHARS], var_name = "c", value_name = "median")
long["characteristic"] = long.c.str.replace("median_", "", n = 1).map(CHARS)
long["group_label"] = long.apply(lambda r: f"G{r.group}: {r.rate_min:.0%}-{r.rate_max:.0%} (n={r.places})", axis = 1)
fig = px.bar(long, x = "group_label", y = "median", facet_col = "characteristic", facet_col_wrap = 3, title = "Place characteristics (median) by storefront match-rate group",
             labels = {"group_label": "match-rate group", "median": ""})
fig.update_yaxes(matches = None, showticklabels = True)
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
show(fig, "place_characteristics_by_match_rate")

### PART C: stores

In [13]:
### PART C, SUBPART I
## Output: [9ci] one row per storefront (the 542 in [6a]) with:
##   matching: "matched", "reason_missing", "covered_months", "open_months", "share_months_covered" (from [6a])
##   cohort: "first_sales_month" (earliest month with positive retail sales by any license operating at the storefront, Jul 2014 on: LCB archive before 2018,
##           store history file from 2018), "cohort" (its year;
##           "2014-2015" grouped, "no sales record"), "cohort_uncertain" (the earliest license also operated at another address, so its first sales may be from
##           there), "first_wslcb_operating" (first dated WSLCB sample with a license operating there; the samples start in Jul 2018)
##   months active: "first_open_month", "last_open_month" (2019-2021, [1e']), "gap_months" (months between the first and last open month that aren't open),
##           "spells" (separate runs of open months), "opened_in_window" / "closed_in_window" (first open month after 2019-01 / last before 2021-12)
##   sales (2019-2021, open months): "sales_months" (open months), "mean_monthly_sales", "share_open_months_with_sales",
##           "sales_gap_months" (open months with no sales between the first and last month with sales; [1e'] itself has almost no gaps, since a license
##           counts as operating in every month between two samples that list it)
##   address: "has_suite" (a unit / suite / building in the WSLCB address), "highway_address" (state route, highway, or milepost), "licenses_ever" (licenses
##           at the address in [1a]), "tradenames_ever", "census_geocode" (the Census geocoder's result for the WSLCB address: exact, non-exact, no match;
##           independent of SafeGraph)
##   surroundings (SafeGraph POIs, current snapshot, around the Census geocoder point, or the [1d] point when the geocoder failed): "pois_50m", "pois_150m",
##           "retail_pois_150m" (NAICS 44-45), "food_pois_150m" (NAICS 722)
##   neighborhood: "place_type", "place_size" (from [6a]), "cbg_pop_density" (2020 block group, people per square mile), "cbg_urban_share"
## Name: 9ci_storefront_characteristics.parquet
stores = coverage[STOREFRONT + ["tradenames", "licenses", "matched", "reason_missing", "open_months", "covered_months", "share_months_covered",
                                "latitude", "longitude", "geo_method", "county_name", "place_fips", "place_name", "place_type", "place_size"]].copy()

# (1) Cohort: first positive sales month of the licenses operating at the storefront
lic_first = sales[sales.sales > 0].groupby("license").month.min().rename("first_sales")
st_lic = store_licenses.merge(lic_first, on = "license", how = "left").merge(lic_n_stores, on = "license")
cohort = (st_lic.sort_values("first_sales")
    .groupby(STOREFRONT).agg(first_sales_month = ("first_sales", "first"), first_license_moves = ("n_storefronts_ever", "first")).reset_index())
cohort["cohort_uncertain"] = cohort.first_license_moves > 1
stores = stores.merge(cohort.drop(columns = "first_license_moves"), on = STOREFRONT, how = "left")
yr = stores.first_sales_month.str[:4]
stores["cohort"] = np.where(yr.notna(), np.where(yr.isin(["2014", "2015"]), "2014-2015", yr), "no sales record")
first_wslcb = locations.groupby(STOREFRONT).first_operating.min().rename("first_wslcb_operating").reset_index()
stores = stores.merge(first_wslcb, on = STOREFRONT, how = "left")

# (2) Months active and gaps, 2019-2021
sm = open_months[STOREFRONT + ["month"]].assign(i = lambda d: d.month.map(month_index)).sort_values(STOREFRONT + ["i"])
sm["new_spell"] = sm.groupby(STOREFRONT).i.diff().fillna(99) != 1
active = sm.groupby(STOREFRONT).agg(first_open_month = ("month", "min"), last_open_month = ("month", "max"), n = ("i", "size"),
                                    span = ("i", lambda x: x.max() - x.min() + 1), spells = ("new_spell", "sum")).reset_index()
active["gap_months"] = active.span - active.n
active["opened_in_window"] = active.first_open_month > "2019-01"
active["closed_in_window"] = active.last_open_month < "2021-12"
stores = stores.merge(active.drop(columns = ["n", "span"]), on = STOREFRONT, how = "left")

# (3) Sales over the storefront's open months in 2019-2021
def sales_gaps(g):  # open months with no sales record between the storefront's first and last month with sales (2019-2021)
    g = g.sort_values("month")
    idx = np.flatnonzero(g.has_sales.values)
    return 0 if len(idx) == 0 else int((~g.has_sales.values[idx[0]:idx[-1] + 1]).sum())
hc_open = history_check[history_check.open_wslcb]
sales_st = hc_open.groupby(STOREFRONT).agg(
    sales_months = ("month", "size"), mean_monthly_sales = ("sales", lambda x: x.fillna(0).mean()),
    share_open_months_with_sales = ("has_sales", "mean")).reset_index()
sales_st = sales_st.merge(hc_open.groupby(STOREFRONT)[["month", "has_sales"]].apply(sales_gaps).rename("sales_gap_months").reset_index(), on = STOREFRONT)
stores = stores.merge(sales_st, on = STOREFRONT, how = "left")

# (4) Address: suite and highway flags from the WSLCB street address, licenses and tradenames at the address, the Census geocoder's result
addr = locations.groupby(STOREFRONT).agg(raw_addresses = ("street_address", lambda x: sorted(set(x))), licenses_ever = ("license", "nunique"),
                                         tradenames_ever = ("tradenames", lambda x: len(set(t for l in x for t in l))), zip5 = ("zip5", "first")).reset_index()
SUITE = r"\b(STE|SUITE|UNIT|BLDG|BUILDING|APT|SPC|SPACE|RM|ROOM|LOT)\b|#|\b\d+[A-Z]?\s*&\s*[A-Z\d]"
HIGHWAY = r"\b(HWY|HIGHWAY|SR|STATE ROUTE|ROUTE|RTE|US|MP|MILEPOST|MILE POST)\b|\bWA-?\d"
addr["has_suite"] = addr.raw_addresses.apply(lambda a: any(re.search(SUITE, x.upper()) for x in a))
addr["highway_address"] = addr.street_address_norm.str.upper().str.contains(HIGHWAY, regex = True)
stores = stores.merge(addr.drop(columns = "raw_addresses"), on = STOREFRONT, how = "left")

geo = pd.read_csv(f"{CENSUS}/wslcb_census_geocoder.csv", header = None, dtype = str,
                  names = ["geocode_id", "input", "match", "match_type", "matched_address", "lonlat", "tiger_id", "side", "state", "county", "tract", "block"])
geo["input"] = geo.input.str.upper().str.strip()
inputs = locations.assign(input = (locations.street_address + ", " + locations.city_norm + ", WA, " + locations.zip5.fillna("")).str.upper().str.strip())
geo = inputs[STOREFRONT + ["input"]].merge(geo, on = "input", how = "inner")
geo["census_geocode"] = np.where(geo.match == "Match", np.where(geo.match_type == "Exact", "exact", "non-exact"), "no match")
geo[["glon", "glat"]] = geo.lonlat.str.split(",", expand = True).astype(float)
rank = {"exact": 0, "non-exact": 1, "no match": 2}
geo = geo.assign(r = geo.census_geocode.map(rank)).sort_values("r").drop_duplicates(STOREFRONT)
stores = stores.merge(geo[STOREFRONT + ["census_geocode", "glat", "glon"]], on = STOREFRONT, how = "left")
print("storefronts with a Census geocoder row:", stores.census_geocode.notna().sum(), "of", len(stores))

# (5) Surrounding businesses: SafeGraph POIs within 50 m / 150 m of the Census geocoder point (or the [1d] point when the geocoder has none)
stores["pt_lat"] = stores.glat.where(stores.glat.notna(), stores.latitude)
stores["pt_lon"] = stores.glon.where(stores.glon.notna(), stores.longitude)
stores["point_source"] = np.where(stores.glat.notna(), "census geocoder", "[1d]: " + stores.geo_method.astype(str))
poi = (pl.scan_parquet(f"{POI_FOLDER}/*.parquet").filter(pl.col("REGION") == "WA")
       .select(["PLACEKEY", "LATITUDE", "LONGITUDE", "NAICS_CODE"]).collect().to_pandas())
poi = gpd.GeoDataFrame(poi, geometry = gpd.points_from_xy(poi.LONGITUDE, poi.LATITUDE), crs = 4326).to_crs(32610)
pts = gpd.GeoDataFrame(stores[STOREFRONT], geometry = gpd.points_from_xy(stores.pt_lon, stores.pt_lat), crs = 4326).to_crs(32610)
for r in [50, 150]:
    j = gpd.sjoin(poi, gpd.GeoDataFrame(pts[STOREFRONT], geometry = pts.buffer(r)), predicate = "within")
    j["naics2"] = j.NAICS_CODE.astype(str).str[:2]
    j["naics3"] = j.NAICS_CODE.astype(str).str[:3]
    agg = j.groupby(STOREFRONT).agg(n = ("PLACEKEY", "size"), retail = ("naics2", lambda x: x.isin(["44", "45"]).sum()),
                                    food = ("naics3", lambda x: (x == "722").sum())).reset_index()
    agg = agg.rename(columns = {"n": f"pois_{r}m", "retail": f"retail_pois_{r}m", "food": f"food_pois_{r}m"})
    stores = stores.merge(agg, on = STOREFRONT, how = "left")
    stores[[f"pois_{r}m", f"retail_pois_{r}m", f"food_pois_{r}m"]] = stores[[f"pois_{r}m", f"retail_pois_{r}m", f"food_pois_{r}m"]].fillna(0)
print("WA SafeGraph POIs:", len(poi))

# (6) Neighborhood: the 2020 block group the [1d] point is in ([1d] cbg2020): density and urban share
bg = gpd.read_file(f"zip://{DOWNLOADS}/tl_2020_53_bg.zip")
bg = pd.DataFrame({"cbg2020": bg.GEOID, "land_sq_mi": bg.ALAND / 2589988.11})
bg_pop = blocks.assign(cbg2020 = blocks.block2020.str[:12], urban_pop = blocks.population_total * blocks.urban).groupby("cbg2020")[["population_total", "urban_pop"]].sum().reset_index()
bg = bg.merge(bg_pop, on = "cbg2020", how = "left")
bg["cbg_pop_density"] = bg.population_total / bg.land_sq_mi.where(bg.land_sq_mi > 0)
bg["cbg_urban_share"] = bg.urban_pop / bg.population_total.where(bg.population_total > 0)
store_cbg = coords.groupby(STOREFRONT).cbg2020.first().reset_index()
stores = stores.merge(store_cbg, on = STOREFRONT, how = "left").merge(bg[["cbg2020", "cbg_pop_density", "cbg_urban_share"]], on = "cbg2020", how = "left")

stores.drop(columns = ["glat", "glon"]).to_parquet("processed-data/9ci_storefront_characteristics.parquet")
print("storefronts:", len(stores), "| cohort uncertain:", stores.cohort_uncertain.sum(), "| no sales record:", (stores.cohort == "no sales record").sum())
display(stores.groupby("matched").point_source.value_counts().unstack(0))

storefronts with a Census geocoder row: 541 of 542


WA SafeGraph POIs: 568297


storefronts: 542 | cohort uncertain: 15 | no sales record: 43


matched,False,True
point_source,,
[1d]: matched_placekey,1.0,2.0
[1d]: openstreetmap,2.0,NaN
[1d]: same_address_placekey,NaN,8.0
[1d]: same_address_poi,8.0,24.0
[1d]: zcta,3.0,2.0
census geocoder,101.0,391.0


In [14]:
### PART C, SUBPART II
## Output: [9cii] matched vs. unmatched storefronts, one row per characteristic: "characteristic", "kind" (share or continuous), "matched_n", "unmatched_n",
##         "matched_mean", "unmatched_mean", "difference" (matched - unmatched), "matched_median", "unmatched_median", "p_value" (Welch t-test for continuous,
##         chi-square / Fisher for shares, rank-sum shown separately as "ranksum_p"), and the means by reason the store is missing
## Name: 9cii_matched_vs_unmatched_storefronts.parquet
d = stores.copy()
d["log_mean_monthly_sales"] = np.log1p(d.mean_monthly_sales)
COHORTS = ["2014-2015", "2016", "2017", "2018", "2019", "2020", "2021", "no sales record"]
for c in COHORTS:
    d[f"cohort_{c}"] = d.cohort == c
d["census_geocode_exact"] = d.census_geocode == "exact"
d["census_geocode_failed"] = d.census_geocode == "no match"
for t in ["city", "CDP", "outside any place"]:
    d[f"place_type_{t}"] = d.place_type.fillna("outside any place").replace({"town": "city"}) == t

SHARES = ([f"cohort_{c}" for c in COHORTS] + ["cohort_uncertain",
          "opened_in_window", "closed_in_window", "has_suite", "highway_address", "census_geocode_exact", "census_geocode_failed",
          "place_type_city", "place_type_CDP", "place_type_outside any place"])
CONT = ["open_months", "gap_months", "spells", "sales_gap_months", "mean_monthly_sales", "log_mean_monthly_sales", "share_open_months_with_sales", "licenses_ever",
        "tradenames_ever", "pois_50m", "pois_150m", "retail_pois_150m", "food_pois_150m", "cbg_pop_density", "cbg_urban_share"]
rows = []
m, u = d[d.matched], d[~d.matched]
for c in SHARES + CONT:
    a, b = m[c].dropna().astype(float), u[c].dropna().astype(float)
    if c in SHARES:
        tab = np.array([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])
        p = stats.fisher_exact(tab).pvalue if tab.min() < 5 else stats.chi2_contingency(tab)[1]
    else:
        p = stats.ttest_ind(a, b, equal_var = False).pvalue
    r = {"characteristic": c, "kind": "share" if c in SHARES else "continuous", "matched_n": len(a), "unmatched_n": len(b),
         "matched_mean": a.mean(), "unmatched_mean": b.mean(), "difference": a.mean() - b.mean(),
         "matched_median": a.median(), "unmatched_median": b.median(), "p_value": p, "ranksum_p": stats.mannwhitneyu(a, b).pvalue}
    for reason, g in u.groupby("reason_missing"):
        r[f"mean_if_{reason}"] = g[c].dropna().astype(float).mean()
    rows.append(r)
compare = pd.DataFrame(rows)
compare.to_parquet("processed-data/9cii_matched_vs_unmatched_storefronts.parquet")
pd.set_option("display.width", 250)
display(compare[["characteristic", "matched_n", "unmatched_n", "matched_mean", "unmatched_mean", "difference", "matched_median", "unmatched_median", "p_value"]].round(3))
print("Unmatched storefronts by reason:", u.reason_missing.value_counts().to_dict())
display(compare[["characteristic"] + [c for c in compare.columns if c.startswith("mean_if_")]].round(3))

## All together: linear probability model of matched on the characteristics, county fixed effects, robust standard errors
reg = d.assign(cohort_c = d.cohort.replace({"no sales record": "none"}), place_t = d.place_type.fillna("outside").replace({"town": "city", "outside any place": "outside"}),
               log_pois_150m = np.log1p(d.pois_150m), log_cbg_density = np.log1p(d.cbg_pop_density), matched_i = d.matched.astype(int),
               has_suite_i = d.has_suite.astype(int), highway_i = d.highway_address.astype(int), exact_i = d.census_geocode_exact.astype(int))
# gap_months is left out: [1e'] has almost no gaps (a handful of storefronts), so its coefficient would rest on those few
formula = ("matched_i ~ C(cohort_c, Treatment('2014-2015')) + open_months + has_suite_i + highway_i + exact_i + licenses_ever"
           " + log_pois_150m + log_cbg_density + C(place_t, Treatment('city'))")
for label, f, data in [("without sales (all storefronts)", formula, reg),
                       ("with sales (2019-2021 open months)", formula + " + log_mean_monthly_sales", reg[reg.sales_months > 0])]:
    for fe in ["", " + C(county_name)"]:
        fit = smf.ols(f + fe, data = data.dropna(subset = ["log_cbg_density"])).fit(cov_type = "HC1")
        print(f"\n=== LPM {label}{', county FE' if fe else ''}: n = {int(fit.nobs)}, R2 = {fit.rsquared:.3f}")
        out = pd.DataFrame({"coef": fit.params, "se": fit.bse, "p": fit.pvalues})
        display(out[~out.index.str.startswith("C(county_name)")].round(3))

,characteristic,matched_n,unmatched_n,matched_mean,unmatched_mean,difference,matched_median,unmatched_median,p_value
0,cohort_2014-2015,427,115,0.363,0.104,0.259,0.00,0.000,0.000
1,cohort_2016,427,115,0.286,0.070,0.216,0.00,0.000,0.000
2,cohort_2017,427,115,0.159,0.087,0.072,0.00,0.000,0.070
3,cohort_2018,427,115,0.105,0.183,-0.077,0.00,0.000,0.037
4,cohort_2019,427,115,0.040,0.070,-0.030,0.00,0.000,0.271
5,cohort_2020,427,115,0.042,0.078,-0.036,0.00,0.000,0.181
6,cohort_2021,427,115,0.002,0.043,-0.041,0.00,0.000,0.002
7,cohort_no sales record,427,115,0.002,0.365,-0.363,0.00,0.000,0.000
8,cohort_uncertain,427,115,0.014,0.078,-0.064,0.00,0.000,0.001
9,opened_in_window,427,115,0.037,0.165,-0.128,0.00,0.000,0.000


Unmatched storefronts by reason: {'no Veraset place at the address': 63, 'only other businesses at the address': 38, 'name only outside its WSLCB months': 10, 'rejected in review': 4}


,characteristic,mean_if_name only outside its WSLCB months,mean_if_no Veraset place at the address,mean_if_only other businesses at the address,mean_if_rejected in review
0,cohort_2014-2015,0.100,0.095,0.105,0.250
1,cohort_2016,0.300,0.048,0.053,0.000
2,cohort_2017,0.200,0.048,0.105,0.250
3,cohort_2018,0.300,0.190,0.132,0.250
4,cohort_2019,0.100,0.079,0.026,0.250
5,cohort_2020,0.000,0.048,0.158,0.000
6,cohort_2021,0.000,0.048,0.053,0.000
7,cohort_no sales record,0.000,0.444,0.368,0.000
8,cohort_uncertain,0.400,0.048,0.053,0.000
9,opened_in_window,0.000,0.143,0.263,0.000



=== LPM without sales (all storefronts): n = 542, R2 = 0.513


,coef,se,p
Intercept,0.053,0.122,0.662
"C(cohort_c, Treatment('2014-2015'))[T.2016]",0.010,0.026,0.687
"C(cohort_c, Treatment('2014-2015'))[T.2017]",-0.049,0.038,0.193
"C(cohort_c, Treatment('2014-2015'))[T.2018]",-0.212,0.054,0.000
"C(cohort_c, Treatment('2014-2015'))[T.2019]",-0.252,0.094,0.007
"C(cohort_c, Treatment('2014-2015'))[T.2020]",-0.075,0.093,0.421
"C(cohort_c, Treatment('2014-2015'))[T.2021]",-0.492,0.133,0.000
"C(cohort_c, Treatment('2014-2015'))[T.none]",-0.714,0.053,0.000
"C(place_t, Treatment('city'))[T.CDP]",0.003,0.033,0.935
"C(place_t, Treatment('city'))[T.outside]",0.065,0.047,0.168



=== LPM without sales (all storefronts), county FE: n = 542, R2 = 0.555


,coef,se,p
Intercept,-0.125,0.204,0.542
"C(cohort_c, Treatment('2014-2015'))[T.2016]",0.012,0.028,0.676
"C(cohort_c, Treatment('2014-2015'))[T.2017]",-0.037,0.041,0.371
"C(cohort_c, Treatment('2014-2015'))[T.2018]",-0.215,0.057,0.000
"C(cohort_c, Treatment('2014-2015'))[T.2019]",-0.238,0.095,0.012
"C(cohort_c, Treatment('2014-2015'))[T.2020]",-0.060,0.092,0.516
"C(cohort_c, Treatment('2014-2015'))[T.2021]",-0.526,0.139,0.000
"C(cohort_c, Treatment('2014-2015'))[T.none]",-0.728,0.057,0.000
"C(place_t, Treatment('city'))[T.CDP]",-0.015,0.034,0.667
"C(place_t, Treatment('city'))[T.outside]",0.064,0.047,0.180



=== LPM with sales (2019-2021 open months): n = 542, R2 = 0.581


,coef,se,p
Intercept,-0.336,0.103,0.001
"C(cohort_c, Treatment('2014-2015'))[T.2016]",0.016,0.024,0.504
"C(cohort_c, Treatment('2014-2015'))[T.2017]",-0.045,0.035,0.196
"C(cohort_c, Treatment('2014-2015'))[T.2018]",-0.173,0.054,0.001
"C(cohort_c, Treatment('2014-2015'))[T.2019]",-0.209,0.082,0.011
"C(cohort_c, Treatment('2014-2015'))[T.2020]",-0.125,0.089,0.157
"C(cohort_c, Treatment('2014-2015'))[T.2021]",-0.501,0.149,0.001
"C(cohort_c, Treatment('2014-2015'))[T.none]",-0.033,0.076,0.664
"C(place_t, Treatment('city'))[T.CDP]",0.019,0.030,0.516
"C(place_t, Treatment('city'))[T.outside]",0.068,0.044,0.119



=== LPM with sales (2019-2021 open months), county FE: n = 542, R2 = 0.618


,coef,se,p
Intercept,-0.517,0.190,0.006
"C(cohort_c, Treatment('2014-2015'))[T.2016]",0.019,0.026,0.457
"C(cohort_c, Treatment('2014-2015'))[T.2017]",-0.032,0.039,0.413
"C(cohort_c, Treatment('2014-2015'))[T.2018]",-0.172,0.055,0.002
"C(cohort_c, Treatment('2014-2015'))[T.2019]",-0.181,0.082,0.027
"C(cohort_c, Treatment('2014-2015'))[T.2020]",-0.101,0.088,0.246
"C(cohort_c, Treatment('2014-2015'))[T.2021]",-0.525,0.145,0.000
"C(cohort_c, Treatment('2014-2015'))[T.none]",-0.002,0.087,0.978
"C(place_t, Treatment('city'))[T.CDP]",0.020,0.031,0.511
"C(place_t, Treatment('city'))[T.outside]",0.066,0.045,0.141


In [15]:
### PART C, CHARTS
coh = d.groupby(["matched", "cohort"]).size().rename("storefronts").reset_index()
coh["share"] = coh.storefronts / coh.groupby("matched").storefronts.transform("sum")
coh["group"] = np.where(coh.matched, "matched", "unmatched")
fig = px.bar(coh, x = "cohort", y = "share", color = "group", barmode = "group", title = "Opening cohort (first month with retail sales), matched vs. unmatched storefronts",
             category_orders = {"cohort": COHORTS})
show(fig, "store_cohort_matched_vs_unmatched")

box = d.melt(id_vars = ["matched"], value_vars = ["mean_monthly_sales", "open_months", "pois_150m", "cbg_pop_density"], var_name = "characteristic")
box["group"] = np.where(box.matched, "matched", "unmatched")
fig = px.box(box, x = "group", y = "value", facet_col = "characteristic", points = "outliers", title = "Matched vs. unmatched storefronts")
fig.update_yaxes(matches = None, showticklabels = True)
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
show(fig, "store_characteristics_matched_vs_unmatched")

In [16]:
### PART C, SUBPART III
## Output: [9ciii] one row per open storefront-month in 2019-2021 ([1e'], the [6a] universe) with "covered" (matched that month), "matched_store",
##         "months_since_first_sales" (from the storefront's first sales month), "sales" and "has_sales" (that month), "year",
##         "edge_month" (within the first or last 3 open months of the storefront's 2019-2021 run), and the storefront characteristics from [9ci]
## Name: 9ciii_storefront_months.parquet
smo = (store_months[STOREFRONT + ["month", "covered"]]
    .merge(history_check[history_check.open_wslcb][STOREFRONT + ["month", "sales", "has_sales"]], on = STOREFRONT + ["month"], how = "left")
    .merge(d[STOREFRONT + ["matched", "first_sales_month", "first_open_month", "last_open_month", "cohort", "has_suite", "highway_address",
                           "pois_150m", "cbg_pop_density", "place_type"]], on = STOREFRONT, how = "left")
    .rename(columns = {"matched": "matched_store"}))
smo["i"] = smo.month.map(month_index)
smo["months_since_first_sales"] = smo.i - smo.first_sales_month.map(lambda x: month_index(x) if isinstance(x, str) else np.nan)
smo["edge_month"] = (smo.i - smo.first_open_month.map(month_index) < 3) | (smo.last_open_month.map(month_index) - smo.i < 3)
smo["year"] = smo.month.str[:4]
smo["in_sales_window"] = smo.month.isin(SALES_MONTHS)
smo.to_parquet("processed-data/9ciii_storefront_months.parquet")

print("storefront-months:", len(smo), "| covered:", smo.covered.mean().round(3))
print("\nCoverage by year:"); display(smo.groupby("year").covered.mean().round(3))
w = smo[smo.in_sales_window]
print("\nCovered vs. not, all storefront-months")
display(w.groupby("covered").agg(storefront_months = ("month", "size"), share_with_sales = ("has_sales", "mean"),
                                 mean_sales = ("sales", lambda x: x.fillna(0).mean()), median_sales_if_any = ("sales", lambda x: x[x > 0].median()),
                                 edge_month = ("edge_month", "mean"), months_since_first_sales = ("months_since_first_sales", "median")).round(3))

## Within-store: matched storefronts with both covered and uncovered months in the sales window, mean sales in each
ws = w[w.matched_store].groupby(STOREFRONT + ["covered"]).sales.apply(lambda x: x.fillna(0).mean()).unstack("covered").dropna()
ws.columns = ["uncovered_mean_sales", "covered_mean_sales"]
print(f"\nWithin-store (storefronts with both kinds of months: {len(ws)}): median covered/uncovered sales ratio",
      round((ws.covered_mean_sales / ws.uncovered_mean_sales.where(ws.uncovered_mean_sales > 0)).median(), 3),
      "| Wilcoxon signed-rank p =", round(stats.wilcoxon(ws.covered_mean_sales, ws.uncovered_mean_sales).pvalue, 4))

## Storefront fixed effects: does a month's sales predict coverage within a storefront? Month fixed effects, clustered by storefront
fe = w[w.matched_store & w.sales.notna()].copy()
fe["log_sales"] = np.log1p(fe.sales.clip(lower = 0))
fe["sid"] = fe.street_address_norm + "|" + fe.city_norm
fe["covered_i"] = fe.covered.astype(int)
fe["edge_i"] = fe.edge_month.astype(int)
fit = smf.ols("covered_i ~ log_sales + edge_i + C(month) + C(sid)", data = fe).fit(cov_type = "cluster", cov_kwds = {"groups": pd.factorize(fe.sid)[0]})
print(f"\nStorefront + month FE, matched storefronts, n = {int(fit.nobs)}:")
display(pd.DataFrame({"coef": fit.params, "se": fit.bse, "p": fit.pvalues}).loc[["log_sales", "edge_i"]].round(4))

## Coverage by sales decile (matched storefronts)
fe["sales_decile"] = pd.qcut(fe.sales.rank(method = "first"), 10, labels = range(1, 11)).astype(int)
dec = fe.groupby("sales_decile").agg(coverage = ("covered", "mean"), median_sales = ("sales", "median")).reset_index()
fig = px.line(dec, x = "sales_decile", y = "coverage", markers = True, hover_data = ["median_sales"],
              title = "Share of storefront-months matched, by that month's sales decile (matched storefronts, 2019-2021)")
show(fig, "store_month_coverage_by_sales_decile")

storefront-months: 17742 | covered: 0.617

Coverage by year:


year
2019    0.533
2020    0.649
2021    0.674
Name: covered, dtype: float64


Covered vs. not, all storefront-months


,storefront_months,share_with_sales,mean_sales,median_sales_if_any,edge_month,months_since_first_sales
covered,,,,,,
False,6791,0.731,151522.205,156479.11,0.214,38.0
True,10951,0.979,270858.947,209890.00,0.164,50.0



Within-store (storefronts with both kinds of months: 261): median covered/uncovered sales ratio 1.082 | Wilcoxon signed-rank p = 0.0



Storefront + month FE, matched storefronts, n = 14293:


,coef,se,p
log_sales,0.0509,0.0171,0.0029
edge_i,0.0443,0.0908,0.6258


In [17]:
### PART C, SUBPART IV
## Output: [9civ] a list to check storefront size by hand: 10 matched and 10 unmatched storefronts (random, seed 9), with the address and a Google Maps link.
##         Columns to fill in by hand: "storefront_size" (small / medium / large / can't tell), "standalone_or_in_strip", "notes"
## Name: review/9c_storefront_size_checklist.csv
pick = pd.concat([d[d.matched].sample(10, random_state = 9), d[~d.matched].sample(10, random_state = 9)])
pick = pick[STOREFRONT + ["zip5", "tradenames", "matched", "reason_missing", "county_name", "pois_50m"]].copy()
pick["tradenames"] = pick.tradenames.apply(lambda x: "; ".join(x) if isinstance(x, (list, np.ndarray)) else x)
pick["google_maps"] = "https://www.google.com/maps/search/?api=1&query=" + (pick.street_address_norm + " " + pick.city_norm + " WA " + pick.zip5.fillna("")).str.replace(" ", "+")
pick[["storefront_size", "standalone_or_in_strip", "notes"]] = ""
pick.to_csv("review/9c_storefront_size_checklist.csv", index = False)
display(pick[STOREFRONT + ["tradenames", "matched", "reason_missing"]])

,street_address_norm,city_norm,tradenames,matched,reason_missing
293,3010 OLD HWY 99 S RD,MOUNT VERNON,CANNAZONE,True,NaN
71,1225 HOLLIS ST,BREMERTON,DESTINATION HIGHWAY 420,True,NaN
45,11401 RAINIER AVE S,SEATTLE,CLOUD CANNABIS; WESTERN BUD,True,NaN
155,17517 15TH AVE NE,SHORELINE,365 RECREATIONAL CANNABIS,True,NaN
441,613 MONTANA AVE,SOUTH BEND,GROWERS OUTLET,True,NaN
506,8630 S MARCH POINT RD,ANACORTES,HIGH SOCIETY; HIGH SOCIETY ANACORTES,True,NaN
375,420 SUNSET BLVD N,RENTON,BUDDY'S,True,NaN
206,20925 CYPRESS WAY,LYNNWOOD,EUPHORIUM 420,True,NaN
117,145 SAMISH WAY,BELLINGHAM,STARBUDS,True,NaN
119,14608 HWY 99,LYNNWOOD,A GREENER TODAY MARIJUANA-LYNNWOOD; PUGET SOUN...,True,NaN


### NOTES
(Written from the outputs above as of the last run. If the inputs change, re-check the numbers.)

**Store history file (Part A)**
- The store history file agrees with the LCB's own archive where they overlap (2018-01 - 2020-08): 99.9% of license-months with sales are in both, with
  sales within 1% (correlation 0.9998). Only 6 archive license-months are missing from it and 74 extra are in it (64 in 2020, the 502data.com years).
- Licenses: 498 are in both the store history file and [1e']; addresses agree for 99.4% of them (3 differ: 413314, 421577, 421667, apparently moves). 37
  are only in the store history file: most closed in 2018 (correctly outside the 2019-2021 universe), but **8 opened in 2021 after the last dated WSLCB
  sample (Apr 2021)**, e.g. 431909 Vibe Cannabis (Kelso), 430876 Yakima Weed Co, 432024 Sea Change Cannabis, so the [1e'] universe misses stores licensed
  after Apr 2021. 75 licenses operating at the 542 by [1e'] are not in the store history file (49 at unmatched storefronts, 26 at matched ones, all but one
  of which have another license in it); **46 of the 542 storefronts have no license in the store history file at all, and 45 of them are unmatched**.
- 88% of open storefront-months in 2019-2021 have positive sales by a license operating there; 12% (2,056 storefront-months, 199 storefronts) have no sales
  record. **50 of the 542 storefronts have no sales in any month of 2019-2021**, and no other license ever at those addresses has sales either, so they look
  like licensed but not (yet) operating storefronts.
- The other direction is small: 19 licenses (145 license-months, 0.9% of license-months with sales, 0.4% of sales) sell in a month when [1e'] doesn't have
  them operating anywhere. Most of these are Apr - Dec 2021 (after the last dated sample); 8 of the 19 have no address among the 542.
- Missing sales are concentrated in the unmatched storefronts: 52% of their open months have no sales record, against 4% for matched storefronts.
- Gating the universe by sales (dropping the 50) would take it from 542 to 492 storefronts (1 matched, 49 unmatched dropped) and the **storefront match rate
  from 78.8% to 86.6%**. Storefront-month coverage goes from 61.7% to 68.4%. With sales through 2021, the newer matched storefronts that had no sales by
  Jul 2020 now have sales, so only 1 matched storefront (17903 HWY 536, Mount Vernon) is dropped.
- 87 license-months have one license operating at two storefronts at once (a move between samples); their sales are split equally.

**Places (Part B)**
- 170 places hold 466 of the 542 storefronts (76 are outside any place). Most places are small: 78 have one storefront, so their match rate is 0 or 1.
  60% of places are at exactly 100% and 17% at 0%, with only 13 distinct values.
- **Quartiles can't be formed without splitting ties**: every version (terciles, quartiles, quintiles) collapses to 3 groups, with the 102 places at 100% in
  the top group. Month coverage (covered / open months) has far fewer ties and gives proper quartiles, so it's the better robustness check.
- **Income, unemployment, and poverty don't differ across match-rate groups** (Kruskal-Wallis p = 0.73, 0.58, 0.25 for match-rate quartiles; 0.93, 0.73,
  0.75 for month-coverage quartiles). Urban share differs a little in the match-rate groups (p = 0.01) but not by month coverage (p = 0.20).
- Population and density do differ, but mechanically: places with many storefronts (the big cities) can hardly be at exactly 0 or 1, so they fill the
  middle group (median population 73k against 7-10k). By month coverage the gap is smaller (p = 0.04), and no characteristic is correlated with the match
  rate among places with 3+ storefronts (all Spearman p > 0.15).
- At the storefront level, which avoids the ties, matched and unmatched storefronts sit in places that look the same (all six characteristics, rank-sum
  p > 0.4).

**Stores (Part C)**
- **Opening cohort** (first month with sales, LCB archive before 2018, store history file from 2018): matched storefronts are older. 36% of matched first
  sold in 2014-2015 against 10% of unmatched, and the 2018 cohort is 10% against 18%. **37% of unmatched storefronts have no sales record at all** (Jul 2014 -
  Dec 2021) against 0.2% of matched, and 4% first sold in 2021.
- **Months active**: unmatched storefronts are open 25 of the 36 months (median 24) against 35 (median 36). 41% of unmatched close and 17% open during
  2019-2021, against 5% and 4% of matched. [1e'] has almost no gap months (a license counts as operating in every month between two samples that list it),
  and gaps in sales are rare and not different (0.21 vs. 0.41 months, p = 0.21).
- **Sales**: median monthly sales are $199k for matched storefronts and $888 for unmatched (mean $253k vs. $53k), and 96% vs. 44% of open months have sales.
- **Address ambiguity doesn't explain it**: suite or unit in the address (44% vs. 44%), highway or route addresses (19% vs. 20%), exact Census geocoder
  match (59% vs. 51%, p = 0.18), number of licenses and tradenames at the address, and businesses within 150 m (median 24 vs. 26) are all about the same.
- **Neighborhood**: unmatched storefronts are somewhat more often in rural or low-density block groups (mean urban share 0.69 vs. 0.83, p = 0.002; median
  block-group density 1,229 vs. 2,324 per square mile), though the place type (city, CDP, outside) is the same.
- **All together** (linear probability model, robust SEs, with and without county fixed effects): cohort, open months, and sales carry the differences
  (+0.06 to 0.07 per log point of mean monthly sales). Address features, surrounding businesses, and place type are all insignificant. Block-group density is
  significant without sales (p = 0.005) but not once sales are in (p = 0.26-0.31).
- **Storefront-months**: coverage rises from 53% (2019) to 65% (2020) and 67% (2021). Covered months almost always have sales (98%) and uncovered ones less
  often (73%). **Within a matched storefront, higher-sales months are somewhat more likely to be covered** (storefront and month fixed effects: +0.051 per log
  point of that month's sales, p = 0.003; covered months have 8% higher sales than the same storefront's uncovered months, Wilcoxon p < 0.001). The effect
  is mostly at the top: coverage is 71-78% across the bottom nine sales deciles and 82% in the top one. The first or last 3 open months don't matter. (With
  sales only through Jul 2020, this was flat: 0.008, p = 0.66.)
- **Storefront size**: deferred. `review/9c_storefront_size_checklist.csv` has 10 matched and 10 unmatched storefronts with Google Maps links to check by hand.

**Takeaway**: the unmatched storefronts are mostly stores that barely operated in 2019-2021 by the store history file's sales records (newer,
shorter-lived, little or no sales; 45 of them have no license in the file at all), not established stores at ambiguous addresses or in different kinds of
places. Gating the store universe by sales removes 49 of the 115 unmatched storefronts and only 1 matched one.

**Open issues**
- Line the sales gate up with Henry's.
- The [1e'] universe misses stores licensed after the last dated WSLCB sample (Apr 2021); at least 8 such licenses sell in 2021.
- Sales are self-reported, and 2020-2021 come from 502data.com captures; only months with positive sales are listed.
- Surrounding businesses come from the current SafeGraph POI file, not 2019-2021.
- ACS place estimates for small CDPs have wide margins of error; 58 places have no median income estimate.